# 01 — Data Generation Validation

Visual validation of synthetic field profiles, host capacities, and costs.
Run this after changes to `src/data_generation/` to verify profiles look realistic.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.config import load_config
from src.data_generation import generate_data

# master_seed=8 matches the median-HVI experiment run.
cfg = load_config('../config/default.yaml', overrides={'master_seed': 8})
field_data, host_data, cost_data = generate_data(cfg)

# Assemble CAPEX matrix using data-generation n_wells (for display)
capex_matrix = cost_data.capex(field_data.n_wells)

years = np.arange(cfg['n_years'])
print(f"Generated {field_data.n_fields} fields, {host_data.n_hosts} hosts, {cfg['n_years']} years")


## Field Oil Production Profiles
Profile shape: ramp up to max well capacity, then exponential decline.
Max rate = n_wells x per_well_rate x complexity_factor.
Plateau duration is emergent (depends on UR vs max rate). Host capacity capping happens downstream.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for i in range(field_data.n_fields):
    ax.plot(years, field_data.oil[i], marker='o', markersize=3, label=field_data.names[i])
ax.set_xlabel('Year')
ax.set_ylabel('Oil production (MSm3/y)')
ax.set_title('Field Oil Production Profiles')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Field Gas Production Profiles
Should correlate with oil (gas = oil × GOR).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for i in range(field_data.n_fields):
    ax.plot(years, field_data.gas[i], marker='o', markersize=3, label=field_data.names[i])
ax.set_xlabel('Year')
ax.set_ylabel('Gas production (BSm3/y)')
ax.set_title('Field Gas Production Profiles')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Field Water Production Profiles
Should rise over time (logistic water cut).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for i in range(field_data.n_fields):
    ax.plot(years, field_data.water[i], marker='o', markersize=3, label=field_data.names[i])
ax.set_xlabel('Year')
ax.set_ylabel('Water production (MSm3/y)')
ax.set_title('Field Water Production Profiles')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Per-Fluid Profiles
Three-panel oil, gas and water production profiles per field.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
panels = [
    (axes[0], field_data.oil, 'Oil (MSm$^3$/y)'),
    (axes[1], field_data.gas, 'Gas (BSm$^3$/y)'),
    (axes[2], field_data.water, 'Water (MSm$^3$/y)'),
]
for ax, data, ylabel in panels:
    for i in range(field_data.n_fields):
        ax.plot(years, data[i], marker='o', markersize=3, label=field_data.names[i])
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Year')
axes[0].legend(loc='upper right', ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## Host Oil Capacity vs Total Field Oil Demand
Hosts should NOT be able to absorb all fields — ensures non-trivial optimization.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Total field demand (with gray shaded envelope to emphasize as the reference line)
total_field_oil = field_data.oil.sum(axis=0)
ax.fill_between(years, total_field_oil, alpha=0.15, color='grey', label='_nolegend_')
ax.plot(years, total_field_oil, 'k--', linewidth=2.5, label='Total field demand')

# Host capacities (lighter to let aggregate lines dominate)
for j in range(host_data.n_hosts):
    ax.plot(years, host_data.oil_capacity[j], linewidth=1.5, label=f'{host_data.names[j]} capacity')

# Total host capacity
total_host_cap = host_data.oil_capacity.sum(axis=0)
ax.plot(years, total_host_cap, 'r-', linewidth=2.5, label='Total host capacity')

ax.set_xlabel('Year')
ax.set_ylabel('Oil (MSm3/y)')
ax.set_title('Host Capacity vs Field Demand (Oil)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Spatial Map — Field and Host Positions
Fields and hosts are placed randomly in a 2D area. Distance between a field and a host drives tieback pipeline cost.

In [ ]:
from collections import defaultdict
from scipy.spatial.distance import cdist

# Distance matrix: (n_fields, n_hosts)
dist_matrix = cdist(field_data.coords, host_data.coords)

# Nearest host per field (K-means cluster assignment)
nearest_host = dist_matrix.argmin(axis=1)

# Compact display names matching the per-portfolio network maps in notebook 11
field_names_short = [f"F{i + 1}" for i in range(field_data.n_fields)]
host_names_short = [f"H{j + 1}" for j in range(host_data.n_hosts)]

fig, ax = plt.subplots(figsize=(8, 8))

# Map extent used for axis padding and perpendicular label offset
all_xy = np.vstack([field_data.coords, host_data.coords])
x_min, x_max = all_xy[:, 0].min(), all_xy[:, 0].max()
y_min, y_max = all_xy[:, 1].min(), all_xy[:, 1].max()
map_extent = max(x_max - x_min, y_max - y_min)
perp_offset = 0.025 * map_extent

# Per-field manual nudge in km for the distance label.
label_nudge_km = {
    2: (0.0, 2.0),  # F3 — push 11 km label up a touch
    3: (0.0, 2.0),  # F4 — push 15 km label up a touch
}

# Per-field override (in points) for the field-name annotation offset.
# Default is (8, 8) — top-right of the marker.
field_name_offset = {
    2: (8, 14),    # F3 — lift name a bit higher
    5: (-18, 6),   # F6 — push name to the left to clear F8
}

# Group fields by assigned host. Within each cluster, sort by angle around the
# host and alternate the perpendicular label offset sign so labels don't pile
# up on the same side when fields are close together.
fields_per_host = defaultdict(list)
for f in range(field_data.n_fields):
    fields_per_host[int(nearest_host[f])].append(f)
for h in fields_per_host:
    hx, hy = host_data.coords[h]
    fields_per_host[h].sort(key=lambda f: np.arctan2(
        field_data.coords[f, 1] - hy,
        field_data.coords[f, 0] - hx,
    ))

# K-means assignment lines with distance labels
for h, fields in fields_per_host.items():
    hx, hy = host_data.coords[h]
    for i, f in enumerate(fields):
        fx, fy = field_data.coords[f]
        ax.plot([fx, hx], [fy, hy], ls='--', color='#555555',
                linewidth=1.3, alpha=0.85, zorder=1)

        t = 0.35
        mx, my = fx + t * (hx - fx), fy + t * (hy - fy)
        dx, dy = hx - fx, hy - fy
        L = float(np.hypot(dx, dy))
        sign = 1.0 if i % 2 == 0 else -1.0
        if L > 0:
            mx += sign * perp_offset * (-dy / L)
            my += sign * perp_offset * (dx / L)
        nudge_dx, nudge_dy = label_nudge_km.get(f, (0.0, 0.0))
        mx += nudge_dx
        my += nudge_dy
        ax.annotate(
            f"{dist_matrix[f, h]:.0f} km",
            (mx, my), fontsize=8, ha='center', va='center', zorder=4,
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='gray', alpha=0.95),
        )

# Fields
ax.scatter(field_data.coords[:, 0], field_data.coords[:, 1],
           s=120, c='steelblue', marker='o', zorder=5,
           edgecolors='white', linewidth=1.5)
for f, name in enumerate(field_names_short):
    offset = field_name_offset.get(f, (8, 8))
    ax.annotate(name, field_data.coords[f], fontsize=9,
                fontweight='bold', textcoords='offset points',
                xytext=offset, color='steelblue')

# Hosts
ax.scatter(host_data.coords[:, 0], host_data.coords[:, 1],
           s=200, c='tomato', marker='s', zorder=5,
           edgecolors='white', linewidth=1.5)
for h, name in enumerate(host_names_short):
    ax.annotate(name, host_data.coords[h], fontsize=9,
                fontweight='bold', textcoords='offset points',
                xytext=(8, 8), color='tomato')

# Square axes centred on the point cloud
half = map_extent / 2 + 0.08 * map_extent
xc, yc = (x_min + x_max) / 2, (y_min + y_max) / 2
ax.set_xlim(xc - half, xc + half)
ax.set_ylim(yc - half, yc + half)

ax.set_xlabel('x (km)')
ax.set_ylabel('y (km)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print('Distance matrix (km):')
print(f'{"":>10}', end='')
for j in range(host_data.n_hosts):
    print(f'{host_data.names[j]:>12}', end='')
print()
for i in range(field_data.n_fields):
    print(f'{field_data.names[i]:>10}', end='')
    for j in range(host_data.n_hosts):
        print(f'{dist_matrix[i, j]:>12.1f}', end='')
    print()

## Summary Table

In [ ]:
print('=== Field Summary ===')
print(f'{"Field":<10} {"Peak Oil":>10} {"Total Oil":>10} {"Peak Gas":>10} {"Peak Water":>10}')
print(f'{"":<10} {"MSm3/y":>10} {"MSm3":>10} {"BSm3/y":>10} {"MSm3/y":>10}')
print('-' * 52)
for i in range(field_data.n_fields):
    print(f'{field_data.names[i]:<10} {field_data.oil[i].max():>10.3f} {field_data.oil[i].sum():>10.1f} '
          f'{field_data.gas[i].max():>10.4f} {field_data.water[i].max():>10.3f}')

print(f'\n=== Host Summary ===')
print(f'{"Host":<10} {"Oil Cap":>10} {"Gas Cap":>10} {"Water Cap":>10} {"OPEX/y":>10} {"Var OPEX":>10} {"ABEX":>10}')
print(f'{"":<10} {"MSm3/y":>10} {"BSm3/y":>10} {"MSm3/y":>10} {"MNOK":>10} {"USD/bbl":>10} {"MNOK":>10}')
print('-' * 72)
for j in range(host_data.n_hosts):
    print(f'{host_data.names[j]:<10} {host_data.oil_capacity[j].mean():>10.2f} '
          f'{host_data.gas_capacity[j].mean():>10.2f} {host_data.water_capacity[j].mean():>10.2f} '
          f'{cost_data.opex_rate[j]:>10.0f} {cost_data.variable_opex_rate[j]:>10.1f} '
          f'{cost_data.abex[j]:>10.0f}')

print(f'\n=== Tieback CAPEX (MNOK) ===')
print(f'{"":>10}', end='')
for j in range(host_data.n_hosts):
    print(f'{host_data.names[j]:>12}', end='')
print()
for i in range(field_data.n_fields):
    print(f'{field_data.names[i]:>10}', end='')
    for j in range(host_data.n_hosts):
        print(f'{capex_matrix[i, j]:>12.0f}', end='')
    print()

print(f'\n=== Optimization Feasibility ===')
total_peak = sum(field_data.oil[i].max() for i in range(field_data.n_fields))
total_cap = sum(host_data.oil_capacity[j].mean() for j in range(host_data.n_hosts))
print(f'Total peak field oil demand: {total_peak:.2f} MSm3/y')
print(f'Total avg host oil capacity: {total_cap:.2f} MSm3/y')
print(f'Demand/capacity ratio: {total_peak/total_cap:.2f} (>1 = non-trivial optimization)')

## Field Reservoir Attributes
New per-field attributes: UR, complexity, depth, n_wells, GOR. These drive both production profiles and CAPEX.

In [ ]:
dg = cfg['data_generation']
barrel = cfg['valuation']['barrel']

print('=== Field Reservoir Attributes ===')
print(f'{"Field":<10} {"UR MSm3":>8} {"UR Mbbl":>8} {"Complex":>8} {"Depth m":>8} {"Wells":>6} {"GOR":>6} {"MaxRate":>8}')
print('-' * 64)
for i in range(field_data.n_fields):
    ur = field_data.ultimate_recovery[i]
    max_rate = field_data.n_wells[i] * dg['per_well_rate'] * field_data.complexity_factor[i]
    print(f'{field_data.names[i]:<10} {ur:>8.1f} {ur * barrel:>8.0f} '
          f'{field_data.complexity_factor[i]:>8.2f} {field_data.depth[i]:>8.0f} '
          f'{field_data.n_wells[i]:>6d} {field_data.gor[i]:>6.0f} {max_rate:>8.2f}')

print(f'\n=== CAPEX Breakdown (cheapest host per field) ===')
print(f'{"Field":<10} {"Drilling":>10} {"Subsea":>10} {"Pipeline":>10} {"Total":>10} {"Host":>6}')
print('-' * 58)
for i in range(field_data.n_fields):
    j_best = capex_matrix[i].argmin()
    drilling = cost_data.well_cost_per_well[i] * field_data.n_wells[i]
    pipeline = cost_data.pipeline_cost[i, j_best]
    subsea = cost_data.subsea_base
    total = drilling + subsea + pipeline
    print(f'{field_data.names[i]:<10} {drilling:>10.0f} {subsea:>10.0f} {pipeline:>10.0f} '
          f'{total:>10.0f} {host_data.names[j_best]:>6}')

print(f'\nCAPEX range: {capex_matrix.min():.0f} – {capex_matrix.max():.0f} MNOK')
print(f'Target range: 4,000 – 40,000 MNOK')